<a href="https://colab.research.google.com/github/krittiyaT/Project-Group2/blob/main/Project_Library_Group2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ขั้นตอนที่ 1 เตรียมข้อมูล
ข้อมูลหนังสือ : กำหนดรหัสหนังสือ(D000-001ถึงD900-015), หมวดหมู่(D000-D900), ชื่อผู้แต่ง, ชื่อหนังสือ

ข้อมูลสมาชิก : กำหนดรหัสสมาชิก(PS0001-PT0050), หมวดหมู่สมาชิก(นักศึกษา,อาจารย์)

##ข้อมูลหลักที่ต้องเก็บ: หนังสือ, สมาชิก, การยืมคืน

##คุณสมบัติ
หนังสือ: รหัสหนังสือ, ชื่อเรื่อง, ชื่อผู้แต่ง, หมวดหมู่, สถานะ(ว่าง/ถูกยืม)

สมาชิก: รหัสสมาชิก, ชื่อ, ประเภทสมาชิก(นักศึกษา/อาจารย์), จำนวนวันที่ยืมได้

การยืม: รหัสการยืม ( L0001-L00350 ) , หนังสือที่ยืม, สมาชิกที่ยืม, วันที่ยืม, กำหนดคืน, วันที่คืนจริง

##หน้าที่

หนังสือ: ยืม, คืน

การยืม: คำนวณค่าปรับ, บันทึกการคืน

##การคำนวณค่าปรับ

ระยะเวลาการยืมต่างกัน นักศึกษายืมได้ 14 วัน อาจารย์ยืมได้ 30 วัน


ค่าปรับตามจำนวนวันที่คืนช้า 5บาทต่อวัน

#หมวดหมู่หนังสือ
​D000 คอมพิวเตอร์ ความรู้ทั่วไป และสารสนเทศ (Computer science, Information & General works)

​D100 ปรัชญา และจิตวิทยา (Philosophy & Psychology)

​D200 ศาสนา (Religion)

​D300 สังคมศาสตร์ (Social sciences)

​D400 ภาษาศาสตร์ (Language)

​D500 วิทยาศาสตร์ (Science)

​D600 เทคโนโลยี และวิทยาศาสตร์ประยุกต์ (Technology)

D​700 ศิลปะ และการบันเทิง (Arts & Recreation)

​D800 วรรณคดี และวรรณกรรม (Literature)

​D900 ประวัติศาสตร์ และภูมิศาสตร์ (History & Geography)

In [ ]:
#นำเข้าฟอนต์กราฟ
!pip install openpyxl
import pandas as pd

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 1. ติดตั้งฟอนต์ภาษาไทยลงในระบบ Colab
!apt-get -y install fonts-thai-tlwg > /dev/null 2>&1

# 2. โหลดไฟล์ฟอนต์เข้า Memory และตั้งค่า font_prop
font_path = "/usr/share/fonts/truetype/tlwg/Laksaman.ttf"
font_prop = fm.FontProperties(fname=font_path)

# 3. ตั้งค่า Default ให้ Matplotlib รู้จักฟอนต์และรองรับเครื่องหมายลบ
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

ขั้นตอนที่ 2 — ออกแบบ Class

In [ ]:
#Import และสร้าง Class
import random
from datetime import datetime, timedelta


class Book:
    def __init__(self, book_id, title, author, category_code, category_name, status="ว่าง"):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.category_code = category_code
        self.category_name = category_name
        self.is_available = (status == "ว่าง")

    def borrow(self):
        """เปลี่ยนสถานะเป็นถูกยืม"""
        self.is_available = False

    def return_book(self):
        """เปลี่ยนสถานะกลับเป็นว่าง"""
        self.is_available = True


class Member:
    def __init__(self, member_id, name, member_type, loan_period_days):
        self.member_id = member_id
        self.name = name
        self.member_type = member_type
        self.loan_period_days = loan_period_days


class BookLoan:
    FINE_PER_DAY = 5

    def __init__(self, loan_id, book, member, borrow_date):
        self.loan_id = loan_id
        self.book = book
        self.member = member
        self.borrow_date = borrow_date

        self.due_date = borrow_date + timedelta(
            days=member.loan_period_days
        )

        self.return_date = None

    def calculate_late_fee(self):
        """คำนวณค่าปรับถ้าคืนช้ากว่ากำหนด"""

        if self.return_date is not None and self.return_date > self.due_date:
            late_days = (self.return_date - self.due_date).days
            return late_days * self.FINE_PER_DAY

        return 0

    def mark_returned(self, return_date):
        """บันทึกว่าคืนหนังสือแล้ว"""

        self.return_date = return_date
        self.book.return_book()

# ขั้นตอนที่ 3 - เขียนฟังก์ชันช่วยงาน (Helper Function)

In [ ]:
import pandas as pd

# ฟังชันก์ load() โหลดข้อมูลจากไฟล์ book.csv
def load_books(csv_path):
    """โหลดหนังสือจากไฟล์ CSV แล้วสร้างเป็น Book object"""

  df = pd.read_csv("/content/books.csv")

  return [
      Book(
          row.book_id,
          row.title,
          row.author,
          row.category_code,
          row.category_name,
          row.status
      )
      for row in df.itertuples(index=False)
  ]

# ฟังชันก์ load() โหลดข้อมูลจากไฟล์ members.csv
def load_members(csv_path):
    """โหลดสมาชิกจากไฟล์ CSV แล้วสร้างเป็น Member object"""

    df = pd.read_csv("/contene/members.csv")

    return [
        Member(
            row.member_id,
            row.name,
            row.member_type,
            row.loan_period_days
        )
        for row in df.itertuples(index=False)
    ]

# ฟังก์ชัน สุ่มวันที่ยืม-คืน
def random_borrow_date(start_date, days_range=300):
    """สุ่มวันที่ยืม"""

    offset = random.randint(0, days_range)

    return start_date + timedelta(days=offset)

def decide_return_outcome(
    never_return_probability=0.20,
    late_probability=0.25
):
    """สุ่มผลการคืนหนังสือ"""

    r = random.random()

    if r < never_return_probability:
        return "not_returned"

    elif r < never_return_probability + late_probability:
        return "late"

    else:
        return "on_time"


def calculate_return_date(due_date, outcome):
    """คำนวณวันที่คืนตามผลการสุ่ม"""

    if outcome == "not_returned":
        return None

    elif outcome == "late":
        return due_date + timedelta(days=random.randint(1, 10))

    else:
        return due_date - timedelta(days=random.randint(0, 5))



# ขั้นตอนที่ 3.1 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

In [ ]:
# เรียก load_books() และ load_members() -> โหลดข้อมูลจาก CSV แล้วแปลงแต่ละแถวเป็น object
test_books = load_books("books.csv")
test_members = load_members("members.csv")

print("จำนวนหนังสือที่โหลดได้:", len(test_books))
print("จำนวนสมาชิกที่โหลดได้:", len(test_members))

print("\nตัวอย่างหนังสือเล่มแรก:")
print("  รหัส:", test_books[0].book_id, "| ชื่อ:", test_books[0].title, "| สถานะว่าง:", test_books[0].is_available)

print("\nตัวอย่างสมาชิกคนแรก:")
print("  รหัส:", test_members[0].member_id, "| ชื่อ:", test_members[0].name,
      "| ประเภท:", test_members[0].member_type, "| ยืมได้:", test_members[0].loan_period_days, "วัน")

In [ ]:
# เรียก random_borrow_date() ซ้ำหลายครั้งด้วย start_date เดียวกัน -> ทุกครั้งควรได้วันที่สุ่มไม่ซ้ำแบบ
demo_start = datetime(2025, 1, 1)

for _ in range(3):
    print("วันที่ยืมที่สุ่มได้:", random_borrow_date(demo_start))


In [ ]:
# เรียก random_borrow_date() ซ้ำหลายครั้งด้วย start_date เดียวกัน -> ทุกครั้งควรได้วันที่สุ่มไม่ซ้ำแบบ
demo_start = datetime(2025, 1, 1)

for _ in range(3):
    print("วันที่ยืมที่สุ่มได้:", random_borrow_date(demo_start))

In [ ]:
# ฟังก์ชัน calculate_return_date() รับ due_date + outcome -> คืนค่าเป็นวันที่คืนจริง (หรือ None ถ้าไม่คืน)
demo_due = demo_start + timedelta(days=14)

print("กำหนดคืน (due_date):", demo_due.strftime("%Y-%m-%d"))
print("ถ้าผลคือ 'not_returned':", calculate_return_date(demo_due, "not_returned"))
print("ถ้าผลคือ 'late'        :", calculate_return_date(demo_due, "late"))
print("ถ้าผลคือ 'on_time'     :", calculate_return_date(demo_due, "on_time"))

**ขั้นตอนที่ 4 — จำลองข้อมูลทีละรายการด้วย Loop**

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

books = pd.read_csv("books.csv", encoding="utf-8-sig").set_index("book_id")
members = pd.read_csv("members.csv", encoding="utf-8-sig").set_index("member_id")

In [ ]:
#3.จำลองการยืมหนังสือ 350 รายการ
random.seed(1)

books = load_books("books.csv")
members = load_members("members.csv")

loans = []

start_date = datetime(2025, 1, 1)


for i in range(1, 351):

    available_books = [
        b for b in books
        if b.is_available
    ]

    if not available_books:

        available_books = books

        for b in available_books:
            b.return_book()

    book = random.choice(available_books)

    member = random.choice(members)

    borrow_date = random_borrow_date(start_date)

    loan_id = f"L{i:04d}"

    loan = BookLoan(
        loan_id,
        book,
        member,
        borrow_date
    )

    book.borrow()

    outcome = decide_return_outcome()

    return_date = calculate_return_date(
        loan.due_date,
        outcome
    )

    if return_date is not None:
        loan.mark_returned(return_date)

    loans.append(loan)


print(f"จำลองการยืมเสร็จแล้วทั้งหมด {len(loans)} รายการ")

ขั้นตอนที่ 4.1 — จำลอง "สมาชิก 1 คนมายืมหนังสือ" แบบ step-by-step

In [ ]:
import time

def simulate_one_loan(loan_id, books, members, start_date, pause=0.0):
    """จำลองขั้นตอนทั้งหมดตอนสมาชิก 1 คนมายืมหนังสือ 1 เล่ม แล้วคืนค่า object BookLoan ที่สร้างเสร็จแล้ว
    """

    print("=" * 60)

    available_books = [b for b in books if b.is_available]
    if not available_books:
        available_books = books
        for b in available_books:
            b.return_book()

    book = random.choice(available_books)
    member = random.choice(members)

    print(f"📚 รายการยืม #{loan_id}: สมาชิก '{member.name}' ({member.member_type}) รหัส {member.member_id} "
      f"เดินมาที่เคาน์เตอร์ ต้องการยืมหนังสือ")
    time.sleep(pause)

    print(f"🔎 พนักงานค้นหาหนังสือว่าง เจอ: '{book.title}' (รหัส {book.book_id}, หมวด {book.category_name})")
    time.sleep(pause)

    borrow_date = random_borrow_date(start_date)
    loan = BookLoan(loan_id, book, member, borrow_date)
    book.borrow()

    print(f"📝 ระบบบันทึกการยืมเรียบร้อย (สถานะหนังสือว่าง = {book.is_available})")
    print(f"   วันที่ยืม: {borrow_date.strftime('%Y-%m-%d')} | "
          f"กำหนดคืน: {loan.due_date.strftime('%Y-%m-%d')} "
          f"({member.loan_period_days} วัน ตามประเภทสมาชิก '{member.member_type}')")
    time.sleep(pause)



    return_date = calculate_return_date(loan.due_date, outcome)

    if return_date is not None:
        loan.mark_returned(return_date)
        print(f"✅ คืนหนังสือแล้วเมื่อ {return_date.strftime('%Y-%m-%d')} "
              f"(สถานะหนังสือกลับเป็นว่าง = {book.is_available})")
    else:
        print("⏳ ยังไม่คืนหนังสือ (สถานะ: not_returned)")

    late_fee = loan.calculate_late_fee()
    print(f"🧾 สรุปรายการ #{loan_id}: {member.name} ยืม '{book.title}' | ค่าปรับ: {late_fee} บาท")

    return loan


# เรียกใช้ทดสอบ 1 ครั้ง ด้วยข้อมูลชุดแยกต่างหาก (ไม่กระทบข้อมูลที่จะใช้จำลอง 350 รายการจริงในขั้นตอนที่ 4 ด้านบน)
demo_books = load_books("books.csv")
demo_members = load_members("members.csv")

demo_loan = simulate_one_loan("DEMO-001", demo_books, demo_members, datetime(2025, 1, 1), pause=0.5)